# Skin Cancer Detection — Colab Kurulumu

Bu notebook, projeyi Google Colab üzerinde GPU ile çalıştırmak için gereken adımları içerir:
1. GPU kontrolü
2. GitHub'dan kodu çekme
3. Bağımlılıkları kurma
4. Kaggle'dan HAM10000 veri setini indirme
5. Manifest oluşturma (build + split + validate)
6. Eğitim
7. Değerlendirme ve görselleştirme
8. Sonuçları kalıcı hale getirme (Drive'a kaydetme)

**Önce yapman gereken:** [kaggle.com/settings](https://www.kaggle.com/settings) → **API** sayfasında kullanıcı adını ve API key'ini (token) hazır bulundur — dosya indirmene gerek yok, 4. adımda ikisini de Colab içinde gizlice gireceksin.

## 1) GPU kontrolü
Çalışma zamanı türünün GPU olduğundan emin ol: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**.

In [ ]:
!nvidia-smi

## 2) Kodu GitHub'dan çek

In [ ]:
!git clone https://github.com/yunus345/skin-cancer-detection.git
%cd skin-cancer-detection

## 3) Bağımlılıkları kur
Colab'da torch/torchvision zaten CUDA destekli kurulu geliyor; `requirements.txt`'teki sürüm belirtilmemiş paketler onları bozmaz, sadece eksik olanları (albumentations, opencv-python vb.) ekler.

In [ ]:
!pip install -q -r requirements.txt

## 4) Kaggle API ile HAM10000'i indir
Aşağıdaki hücre senden **kullanıcı adını** (normal yazılır) ve **API key**'ini (`getpass` ile, ekranda görünmeden) isteyecek — kaggle.json dosyası indirmene gerek yok, kaggle kütüphanesi bu ortam değişkenlerini otomatik okur.

Not: `kmader/skin-cancer-mnist-ham10000` Kaggle'daki standart HAM10000 paylaşımı. Hücre hata verirse [kaggle.com](https://www.kaggle.com)'da "HAM10000" arayıp doğru dataset slug'ını (kullanıcı/dataset-adı) bulup aşağıdaki komutta değiştir.

In [ ]:
import os, getpass
os.environ['KAGGLE_USERNAME'] = input('Kaggle kullanıcı adın: ')
os.environ['KAGGLE_KEY'] = getpass.getpass('Kaggle API key (gizli, ekranda görünmez): ')

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/raw --unzip

## 5) Manifest oluştur (build → split → validate)
`build_manifest.py`, `data/` altındaki tüm görselleri ve CSV metadata dosyalarını otomatik eşleştirir — klasör yapısının birebir yerelle aynı olması gerekmez.

In [ ]:
!python -m src.data.build_manifest --data-root data --out data/processed/manifest.csv
!python -m src.data.split_manifest --manifest data/processed/manifest.csv --out data/processed/manifest.csv
!python -m src.data.validate_manifest --manifest data/processed/manifest.csv

## 6) Eğitim
`finetune-epochs` ve `patience`'ı yerel denemelerimizden daha yüksek tuttuk (model hâlâ iyileşiyordu, erken kesmeyelim). `--device auto` burada CUDA'yı (Colab GPU'su) otomatik seçecek — MPS/SSL ile ilgili hiçbir ek ayar gerekmez (Colab Linux ortamında sistem sertifikaları zaten düzgün çalışır).

In [ ]:
!python -m src.training.train \
  --feature-extract-epochs 5 \
  --finetune-epochs 20 \
  --patience 6 \
  --device auto

## 7) Test seti değerlendirmesi ve grafikler

In [ ]:
!python -m src.evaluation.evaluate --device auto
!python -m src.evaluation.plot_history
!python -m src.evaluation.plot_confusion_matrix

In [ ]:
from IPython.display import Image, display
display(Image('models/training_curves.png'))
display(Image('models/confusion_matrix.png'))

## 8) Sonuçları kalıcı hale getir
**Önemli:** Colab çalışma zamanı kapanınca `models/` klasöründeki her şey (checkpoint'ler, grafikler, metrikler) silinir. Devam etmeden önce Drive'a kopyala.

Aşağıdaki hücre iki kopya oluşturur:
- **`latest/`** — her zaman aynı isim, üzerine yazılır. En son sonucu ararken tarih hatırlamana gerek kalmaz, Drive'da hep aynı yerde.
- **tarih-saat damgalı klasör** — geçmiş koşuları karşılaştırmak istersen diye arşivde kalır, hiçbir zaman silinmez/üzerine yazılmaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import datetime

BASE = '/content/drive/MyDrive/skin-cancer-detection-runs'
os.makedirs(BASE, exist_ok=True)

# Kolay bulunsun diye: her zaman ayni isimde bir "latest" klasorune kaydet (uzerine yazilir)
!rm -rf "{BASE}/latest"
!cp -r models "{BASE}/latest"

# Gecmisi kaybetmemek icin: tarih-saat damgali ayri bir arsiv kopyasi da tut
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
!cp -r models "{BASE}/{stamp}"

print(f"En son sonuc:  {BASE}/latest")
print(f"Arsiv kopyasi: {BASE}/{stamp}")